In [1]:
from GradientGang.TheFreePirate.Optimizer.OptunaOptimizer import OptunaOptimizer
from GradientGang.TheFreePirate.Optimizer.HyperParameter.Hyperparameter import *

c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataParams = {
    'folderPath': '../../dataset/PirateProcessed',
    'trainTimeSeriesFileName': 'pirate_pain_train.csv',
    'trainGlobalFeaturesFileName': 'train_global_features.csv',
    'trainLabelsFileName': 'pirate_pain_train_labels.csv',
    'testTimeSeriesFileName': 'pirate_pain_test.csv',
    'testGlobalFeaturesFileName': 'test_global_features.csv',
    'labelsMapping': {'no_pain': 0, 'low_pain': 1, 'high_pain': 2},
    'batch_size': 64,
    'num_workers': 0,
    'columnsToIgnore': ['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_25', 'joint_26', 'joint_00', 'joint_02', 'joint_03', 'joint_05', 'joint_06'],
}

callbacksParams = {
    'baseLogDir': 'PirateLogs',
    'earlyStoppingPatience': 20,
    'maxEpochs': 1,
}

In [7]:
numLayers = GlobalValuesHyperparameter(IntHyperparameter("globalFeaturesEncoderNumLayers", 1, 2))
embedding = GlobalValuesHyperparameter(CategoricalHyperparameter("globalFeaturesEmbeddingDim", [32, 64]))
dropout = GlobalValuesHyperparameter(FloatHyperparameter("globalFeaturesDropout", 0.0, 0.5))
useGlobal = GlobalHyperparameter( 
    hyperparameter=CategoricalHyperparameter("useGlobalFeatures", [False]),
    values=[
        numLayers,
        embedding,
        dropout
    ]
)

phi = {
    'dataParams': {
        'numFolds': 2,
        'includeTestInFolds': True,
        'augmentTestSet': True,
        'dataAugmentationParams':{
            'nCopies': 2,
            'keepOriginal': True,
            'scaleRange': 0.1,
            'jitterStdDev': 0.05,
            'offsetRange': 0.1,
            'maxWarpFraction': FloatHyperparameter("maxWartFraction", 0.0, 0.2),
            'windowSize': 10,
            'windowStride': 5,
        }
    },
    'modelParams': {
        'f1AverageStrategy': 'weighted',
        'numClasses': 3,
        'reconstructionLossWeight': 0.5,
        'useGlobalFeatures': useGlobal,
        'globalFeaturesEncoderNumLayers': numLayers,
        'globalFeaturesEmbeddingDim': embedding,
        'globalFeaturesDropout': dropout,
        'learningRate': 1e-3,
        'weightDecay': 1e-2,
        'activationFunction': 'relu',
        'timeSeriesEncoderNumLayers': 2,
        'timeSeriesEmbeddingDim': 64,
        'timeSeriesDropout': FloatHyperparameter("timeSeriesDroput", 0.0, 0.2),
        'predictorNumLayers': 2,
        'predictorDropout': 0.1,
        'aggregationStrategyForClassification': 'majorityVoting',
        'predictorFixALogit': False
    }
}

In [8]:
optim = OptunaOptimizer("test", dataParams, callbacksParams, phi)

[I 2025-11-17 18:53:12,543] A new study created in memory with name: no-name-0b779615-eaa5-4697-ae44-96aed78b1839
Seed set to 42


In [9]:
optim.optimize(1)

[PirateDataModule] Setting up K-Folds(2) with data augmentation...
[PirateDataModule] K-Folds setup completed with 2 folds.
[PirateDataModule] Setting up K-Folds(2) with data augmentation...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[PirateDataModule] K-Folds setup completed with 2 folds.

=== Evaluating Fold 1/2 ===
                                                                           

c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 78/78 [00:03<00:00, 22.04it/s, v_num=13, train/total_loss=0.424, train/f1_score=1.000, val/total_loss=0.733, val/f1_score=0.766]

...

Epoch 0: 100%|██████████| 78/78 [00:03<00:00, 23.80it/s, v_num=8, train/total_loss=1.210, train/f1_score=0.853, val/total_loss=0.728, val/f1_score=0.792]

`Trainer.fit` stopped: `max_epochs=1` reached.


Validation DataLoader 0: 100%|██████████| 16/16 [00:00<00:00, 126.45it/s]

[I 2025-11-17 18:53:23,613] Trial 0 finished with value: 0.7793457210063934 and parameters: {'maxWartFraction': 0.10976270078546496, 'useGlobalFeatures': False, 'timeSeriesDroput': 0.1430378732744839}. Best is trial 0 with value: 0.7793457210063934.



Fold 1 Evaluation: 0.7922


In [10]:
optim.getBestModel()

{'dataParams': {'numFolds': 2,
  'includeTestInFolds': True,
  'augmentTestSet': True,
  'dataAugmentationParams': {'nCopies': 2,
   'keepOriginal': True,
   'scaleRange': 0.1,
   'jitterStdDev': 0.05,
   'offsetRange': 0.1,
   'maxWarpFraction': 0.10976270078546496,
   'windowSize': 10,
   'windowStride': 5}},
 'modelParams': {'f1AverageStrategy': 'weighted',
  'numClasses': 3,
  'reconstructionLossWeight': 0.5,
  'useGlobalFeatures': False,
  'globalFeaturesEncoderNumLayers': None,
  'globalFeaturesEmbeddingDim': None,
  'globalFeaturesDropout': None,
  'learningRate': 0.001,
  'weightDecay': 0.01,
  'activationFunction': 'relu',
  'timeSeriesEncoderNumLayers': 2,
  'timeSeriesEmbeddingDim': 64,
  'timeSeriesDropout': 0.1430378732744839,
  'predictorNumLayers': 2,
  'predictorDropout': 0.1,
  'aggregationStrategyForClassification': 'majorityVoting',
  'predictorFixALogit': False}}